# INCRT-geo v9' — pretraining on multi-vertebrate Ensembl proteomes (~200k seq)

This notebook is a **scaling experiment**: same architecture and hyperparameters as v9, but pretrained on a corpus of **eight vertebrate reference proteomes** from Ensembl (~200k sequences after filtering) instead of the human reference proteome alone (~20k). The motivation is to test whether the gap to ESM-2 small reported in the CIBB short paper is dominated by (a) **architectural choices** of INCRT-geo, or (b) **pretraining-corpus scale**, while keeping the framing of "reference proteomes" rather than curated sequence-cluster databases (UniRef, BFD).

## Species selection

Eight vertebrate reference proteomes (Ensembl release 115), spanning ~450 million years of evolutionary divergence:

| Species (Latin)         | Common name        | Distance from human |
|-------------------------|--------------------|---------------------|
| *Homo sapiens*          | human              | 0                   |
| *Macaca mulatta*        | rhesus macaque     | ~25 Myr             |
| *Mus musculus*          | mouse              | ~90 Myr             |
| *Rattus norvegicus*     | rat                | ~90 Myr             |
| *Bos taurus*            | cow                | ~95 Myr             |
| *Gallus gallus*         | chicken            | ~320 Myr            |
| *Xenopus tropicalis*    | tropical clawed frog | ~360 Myr          |
| *Danio rerio*           | zebrafish          | ~450 Myr            |

For each species the file `pep.all.fa.gz` is downloaded (canonical sequences and alternative isoforms). The combined corpus is then **deduplicated by exact sequence string** (so sequences identical between species are kept only once), filtered for non-standard residues and length ≤ 510 amino acids, and used directly for masked language modelling.

## Differences from v9

- Pretraining corpus: 8 vertebrate proteomes (~200k seq) instead of human-only (~20k seq)
- Output paths: `phase11_pretrain_v9p/` and `phase12_pfam_v9p/` (separate from v9, v10, v11)
- Status file: `pipeline_status_v9p.json`

**Everything else is identical to v9:** 1-mer tokeniser, n=512, three-level growth enabled, λ_α=10, λ_geo=1e-5, α_crit=0.9, AdamW with cosine LR, 10 epochs, soft identity init at Level-3 events, `add_param_group` Adam-preserving fix.

## Time budget and resume strategy

Multi-vertebrate pretraining is approximately 10× longer per epoch than human-only pretraining. Expected: **6-8 hours per epoch on Colab Pro+ A100**, so **60-80 hours total** for 10 epochs. Colab Pro+ session limit is ~24h, so the run **will be interrupted multiple times**. Per-epoch checkpointing is enabled; the launcher cell auto-detects the latest `incrt_geo_epoch{N}.pt` and resumes from epoch N+1.

## Robustness against Drive disconnects

Google Drive sometimes disconnects mid-session (`OSError [Errno 107]`). Two tiers of cache are used:

- **Local cache** (`/content/multi_vertebrate_proteins.json`): always available on the Colab RAM disk; survives Drive disconnects but is lost on session restart.
- **Drive cache** (best-effort): copied from local after parsing. If the Drive write fails, training continues; only consequence is a fresh ~5 min reparse if the session restarts.

If Drive disconnects during the run, in a fresh cell run:

```python
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
```

then re-run the Phase 11 cell.

## Predictions (registered before the run)

Three regimes are possible on Pfam-50 linear-probe test accuracy:

- **v9' ≥ 70%** → INCRT-geo on multi-vertebrate corpus is competitive with ESM-2 small (71.13%). The gap reported in CIBB was **mostly due to corpus scale**, not architecture. Strong positive result.
- **v9' ∈ [65%, 70%)** → Substantial improvement over human-only v9 (63.78%) but still below ESM-2 small. Architecture and scale both contribute. Honest mixed result.
- **v9' < 65%** → Multi-vertebrate scaling does not help meaningfully. The gap is **architectural**. Strong negative result, important to report.


In [ ]:
!pip install -q biopython tqdm scikit-learn


In [ ]:
import os, json, gzip, urllib.request, random, re, time, shutil
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt, log, ceil
from dataclasses import dataclass
from typing import List, Optional, Dict, Tuple
from collections import Counter
from tqdm import tqdm

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}, '
          f'Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Mount Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_OK = True
except Exception as e:
    print(f'Drive unavailable: {e}')
    DRIVE_OK = False

# Pipeline directories — V9 uses its own status file and output directories
# to keep results separate from v8/v6 pipeline runs.
if DRIVE_OK:
    PIPELINE_ROOT = '/content/drive/MyDrive/incrt_geo_pipeline'
else:
    PIPELINE_ROOT = './incrt_geo_pipeline'

PHASE11_DIR = os.path.join(PIPELINE_ROOT, 'phase11_pretrain_v9p')
PHASE12_DIR = os.path.join(PIPELINE_ROOT, 'phase12_pfam_v9p')
STATUS_FILE = os.path.join(PIPELINE_ROOT, 'pipeline_status_v9p.json')
DATA_DIR = './data'

for d in [PIPELINE_ROOT, PHASE11_DIR, PHASE12_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Pipeline root: {PIPELINE_ROOT}')
print(f'Phase 11 (pretrain v9p, SwissProt): {PHASE11_DIR}')
print(f'Phase 12 (pfam v9p):                {PHASE12_DIR}')

def load_status():
    if os.path.exists(STATUS_FILE):
        with open(STATUS_FILE) as f: return json.load(f)
    return {'phase11_done': False, 'phase12_done': False, 'last_completed_epoch': 0, 'log': []}

def save_status(s):
    with open(STATUS_FILE, 'w') as f: json.dump(s, f, indent=2)

def log_event(msg):
    print(f'>>> {msg}')
    s = load_status()
    s['log'].append(f'{time.strftime("%Y-%m-%d %H:%M:%S")} — {msg}')
    save_status(s)

status = load_status()
print(f"V9' pipeline status: phase11={status.get('phase11_done', False)}, "
      f"phase12={status.get('phase12_done', False)}, "
      f"last_completed_epoch={status.get('last_completed_epoch', 0)}")


## Common utilities (tokenizer, FASTA loader, model definitions)

In [ ]:
# ── Constants ─────────────────────────────────────────────
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
NON_STANDARD = set('XBZUO*')

# ── Tokenizer ─────────────────────────────────────────────
class ProteinKmerTokenizer:
    """v9: defaults to k=1 (single amino acid). Vocab = 20 AA + 5 special = 25."""
    SPECIAL = ['[PAD]','[MASK]','[CLS]','[SEP]','[UNK]']
    def __init__(self, k=1, alphabet=AA_ALPHABET):
        self.k = k
        self.vocab = list(self.SPECIAL)
        self._build(alphabet, k, '')
        self.t2i = {t:i for i,t in enumerate(self.vocab)}
        self.pad_id  = self.t2i['[PAD]']; self.mask_id = self.t2i['[MASK]']
        self.cls_id  = self.t2i['[CLS]']; self.sep_id  = self.t2i['[SEP]']
        self.unk_id  = self.t2i['[UNK]']
        self.vocab_size = len(self.vocab)
    def _build(self, a, k, p):
        if k == 0: self.vocab.append(p); return
        for c in a: self._build(a, k-1, p+c)
    def encode(self, seq, max_length):
        toks = [seq[i:i+self.k] for i in range(len(seq) - self.k + 1)]
        ids = [self.t2i.get(t, self.unk_id) for t in toks]
        ids = [self.cls_id] + ids[:max_length-2] + [self.sep_id]
        if len(ids) < max_length:
            ids = ids + [self.pad_id]*(max_length - len(ids))
        return ids

# ── Multi-vertebrate Ensembl loader ───────────────────────
# Eight vertebrate reference proteomes from Ensembl release 115.
# We use directory listing to find the actual `*.pep.all.fa.gz` filename for
# each species, since assembly versions in filenames change between releases.

ENSEMBL_RELEASE = 115
ENSEMBL_FTP_BASE = f'https://ftp.ensembl.org/pub/release-{ENSEMBL_RELEASE}/fasta'

# species_dir on Ensembl FTP -> human-readable label
SPECIES = [
    ('homo_sapiens',     'Homo sapiens',       'human'),
    ('macaca_mulatta',   'Macaca mulatta',     'macaque'),
    ('mus_musculus',     'Mus musculus',       'mouse'),
    ('rattus_norvegicus','Rattus norvegicus',  'rat'),
    ('bos_taurus',       'Bos taurus',         'cow'),
    ('gallus_gallus',    'Gallus gallus',      'chicken'),
    ('xenopus_tropicalis','Xenopus tropicalis','xenopus'),
    ('danio_rerio',      'Danio rerio',        'zebrafish'),
]

# Local cache (always available, survives Drive disconnects)
PROTEIN_CACHE_PATH_LOCAL = '/content/multi_vertebrate_proteins.json'
# Drive cache (best-effort)
PROTEIN_CACHE_PATH_DRIVE = (
    os.path.join(PIPELINE_ROOT, 'multi_vertebrate_proteins.json')
    if DRIVE_OK
    else os.path.join(DATA_DIR, 'multi_vertebrate_proteins.json')
)

def list_pep_filename(species_dir):
    """List the FTP directory and find the `*.pep.all.fa.gz` filename for a species."""
    import re as _re
    url = f'{ENSEMBL_FTP_BASE}/{species_dir}/pep/'
    print(f'  Listing {url}...', end=' ', flush=True)
    try:
        with urllib.request.urlopen(url, timeout=30) as r:
            html = r.read().decode('utf-8', errors='ignore')
    except Exception as e:
        print(f'FAILED ({e})')
        return None
    matches = _re.findall(r'href="([^"]+\.pep\.all\.fa\.gz)"', html)
    if not matches:
        print('NO MATCH')
        return None
    fname = matches[0]
    print(f'OK -> {fname}')
    return fname

def download_one_species(species_dir, target_dir):
    """Download a single species' pep.all.fa.gz from Ensembl into target_dir.
    Returns the local path or None on failure. Skips download if already present."""
    fname = list_pep_filename(species_dir)
    if fname is None:
        return None
    local_path = os.path.join(target_dir, fname)
    if os.path.exists(local_path) and os.path.getsize(local_path) > 100_000:
        print(f'    Already downloaded: {local_path} ({os.path.getsize(local_path)/1e6:.1f} MB)')
        return local_path
    url = f'{ENSEMBL_FTP_BASE}/{species_dir}/pep/{fname}'
    print(f'    Downloading {url} ...', end=' ', flush=True)
    try:
        urllib.request.urlretrieve(url, local_path)
        print(f'OK ({os.path.getsize(local_path)/1e6:.1f} MB)')
        return local_path
    except Exception as e:
        print(f'FAILED ({e})')
        # Cleanup partial
        if os.path.exists(local_path):
            try: os.remove(local_path)
            except: pass
        return None

def parse_species_fasta(path, species_label, max_length=510):
    """Parse one species FASTA. Returns list of sequence strings.
    Filters: only standard amino acids, length 10..max_length."""
    seqs = []
    cur_chunks = []
    cur_in_record = False
    n_total = 0; n_kept = 0; n_too_long = 0; n_nonstd = 0; n_too_short = 0

    def _flush():
        nonlocal n_total, n_kept, n_too_long, n_nonstd, n_too_short
        if not cur_in_record:
            return
        n_total += 1
        seq = ''.join(cur_chunks)
        if len(seq) < 10:
            n_too_short += 1
            return
        if len(seq) > max_length:
            n_too_long += 1
            return
        if any(c in NON_STANDARD for c in seq):
            n_nonstd += 1
            return
        seqs.append(seq)
        n_kept += 1

    opener = gzip.open if path.endswith('.gz') else open
    with opener(path, 'rt') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line:
                continue
            if line.startswith('>'):
                _flush()
                cur_in_record = True
                cur_chunks = []
            else:
                cur_chunks.append(line)
        _flush()

    print(f'    {species_label}: total={n_total:,} | kept={n_kept:,} | '
          f'too_long={n_too_long:,} | nonstd={n_nonstd:,} | too_short={n_too_short:,}')
    return seqs

def load_multi_vertebrate_proteins(force_reparse=False):
    """Load the eight-species multi-vertebrate corpus with two-level cache.

    Strategy:
      1. Try local /content/ cache first (always available if it exists)
      2. Fall back to Drive cache (may fail if Drive disconnected)
      3. Otherwise download all 8 species, parse, deduplicate by exact sequence,
         and write both caches (Drive write is best-effort)
    """
    # Tier 1: local cache
    if not force_reparse and os.path.exists(PROTEIN_CACHE_PATH_LOCAL):
        print(f'Loading cached multi-vertebrate proteins from local {PROTEIN_CACHE_PATH_LOCAL}...')
        with open(PROTEIN_CACHE_PATH_LOCAL) as f:
            seqs = json.load(f)
        # Backward-compat: old cache may be (a) a list of strings, or (b) a flat dict
        # {pid: sequence_string}. Promote both to {pid: {'seq': sequence_string}}.
        if isinstance(seqs, list):
            print(f'  Cache was a list ({len(seqs):,} entries); promoting to dict-of-dicts.')
            n_pad = len(str(len(seqs)))
            seqs = {f'mv_{i:0{n_pad}d}': {'seq': s} for i, s in enumerate(seqs)}
            _rewrite = True
        elif isinstance(seqs, dict) and len(seqs) > 0:
            first_val = next(iter(seqs.values()))
            if isinstance(first_val, str):
                print(f'  Cache was a flat dict ({len(seqs):,} entries); promoting to dict-of-dicts.')
                seqs = {pid: {'seq': s} for pid, s in seqs.items()}
                _rewrite = True
            else:
                _rewrite = False
        else:
            _rewrite = False
        if _rewrite:
            try:
                with open(PROTEIN_CACHE_PATH_LOCAL, 'w') as f:
                    json.dump(seqs, f)
                print(f'  Local cache rewritten in correct format.')
            except Exception as e:
                print(f'  WARNING: could not rewrite local cache ({e}); will re-promote each session.')
        print(f'  Loaded {len(seqs):,} sequences from local cache.')
        return seqs

    # Tier 2: Drive cache
    if not force_reparse:
        try:
            if os.path.exists(PROTEIN_CACHE_PATH_DRIVE):
                print(f'Loading cached multi-vertebrate proteins from Drive {PROTEIN_CACHE_PATH_DRIVE}...')
                with open(PROTEIN_CACHE_PATH_DRIVE) as f:
                    seqs = json.load(f)
                # Backward-compat: list or flat dict -> dict-of-dicts.
                if isinstance(seqs, list):
                    print(f'  Drive cache was a list ({len(seqs):,} entries); promoting to dict-of-dicts.')
                    n_pad = len(str(len(seqs)))
                    seqs = {f'mv_{i:0{n_pad}d}': {'seq': s} for i, s in enumerate(seqs)}
                elif isinstance(seqs, dict) and len(seqs) > 0:
                    first_val = next(iter(seqs.values()))
                    if isinstance(first_val, str):
                        print(f'  Drive cache was a flat dict ({len(seqs):,} entries); promoting to dict-of-dicts.')
                        seqs = {pid: {'seq': s} for pid, s in seqs.items()}
                print(f'  Loaded {len(seqs):,} sequences from Drive, copying to local for next time...')
                try:
                    with open(PROTEIN_CACHE_PATH_LOCAL, 'w') as f:
                        json.dump(seqs, f)
                    print(f'  Local mirror written.')
                except Exception as e:
                    print(f'  Local mirror failed ({e}); using Drive load only.')
                return seqs
        except Exception as e:
            print(f'  Drive cache read failed ({e}); will re-parse.')

    # Tier 3: download + parse
    print('Downloading and parsing the 8-species Ensembl corpus.')
    print(f'Ensembl release: {ENSEMBL_RELEASE}')
    all_seqs = []
    per_species_counts = []
    for species_dir, latin, label in SPECIES:
        print(f'  [{label}]')
        fasta_path = download_one_species(species_dir, DATA_DIR)
        if fasta_path is None:
            print(f'  WARNING: skipping {label} (download failed)')
            per_species_counts.append((label, latin, 0))
            continue
        species_seqs = parse_species_fasta(fasta_path, latin, max_length=510)
        per_species_counts.append((label, latin, len(species_seqs)))
        all_seqs.extend(species_seqs)

    # Deduplicate by exact sequence string
    print(f'\n  Total sequences before deduplication: {len(all_seqs):,}')
    deduped_seqs_list = list(set(all_seqs))
    n_dup_removed = len(all_seqs) - len(deduped_seqs_list)
    print(f'  Duplicate sequences removed: {n_dup_removed:,}')
    print(f'  Final corpus size: {len(deduped_seqs_list):,}')

    # Wrap as dict {synthetic_id: {'seq': sequence}} so the dataset class
    # (which accesses self.proteins[pid]['seq']) receives the right structure.
    # Synthetic IDs are zero-padded sequential indices.
    n_pad = len(str(len(deduped_seqs_list)))
    deduped_seqs = {f'mv_{i:0{n_pad}d}': {'seq': s} for i, s in enumerate(deduped_seqs_list)}

    print(f'\n  Per-species kept counts (before deduplication):')
    for label, latin, count in per_species_counts:
        pct = 100*count/max(len(all_seqs),1)
        print(f'    {label:12s} ({latin:24s}): {count:>7,d}  ({pct:5.1f}%)')

    # Write local cache (always works)
    print(f'\nCaching to local {PROTEIN_CACHE_PATH_LOCAL}...')
    try:
        with open(PROTEIN_CACHE_PATH_LOCAL, 'w') as f:
            json.dump(deduped_seqs, f)
        print(f'  Local cache written ({os.path.getsize(PROTEIN_CACHE_PATH_LOCAL)/1e6:.1f} MB).')
    except Exception as e:
        print(f'  WARNING: local cache write failed ({e}). Will re-parse on next call.')

    # Best-effort Drive copy
    try:
        print(f'Copying to Drive cache {PROTEIN_CACHE_PATH_DRIVE}...')
        import shutil as _shutil
        _shutil.copy(PROTEIN_CACHE_PATH_LOCAL, PROTEIN_CACHE_PATH_DRIVE)
        print(f'  Drive cache written.')
    except Exception as e:
        print(f'  WARNING: Drive cache write failed ({e}); local cache only.')
        print(f'           This is non-fatal. Pretraining will proceed.')

    return deduped_seqs


In [ ]:
# ── Model definitions (shared across phases) ────────────
@dataclass
class INCRTConfig:
    d: int = 256
    dk: int = 64
    dv: int = 64
    n: int = 512        # v9: longer sequences
    vocab_size: int = 25  # v9: 1-mer, 20 AA + 5 special
    sigma_a: float = 0.01
    sigma_V_star: float = 1.0
    eta_plus: float = None
    eta_minus: float = None
    kappa: float = 0.5
    kappa_T: float = 2.0
    eps_conv: float = 0.05
    theta_w: float = None
    phi_w: float = None
    phi_g: float = None
    theta_d: float = None
    eps_cone: float = 0.01
    delta_disp: float = None
    T_prune: int = 100
    enable_level3: bool = True
    alpha_crit: float = 0.9
    T_wnd: int = 200
    eps_alpha: float = 0.01
    max_layers: int = 6
    layer3_dwell: int = 500
    lambda_geo: float = 1e-5
    lambda_warmup_steps: int = 2000
    use_log_geo: bool = True
    lambda_alpha: float = 10.0

    def __post_init__(self):
        if self.eta_plus is None: self.eta_plus = 1.0/(10*self.d)
        if self.eta_minus is None: self.eta_minus = self.eta_plus/2
        self.sigma_V_star = sqrt(self.dk * self.n / (self.d * self.dv))

    def calibrate(self, gamma_res, X_norm):
        if hasattr(gamma_res, '__len__'):
            gamma_res = float(np.median(gamma_res))
        if hasattr(X_norm, '__len__'):
            X_norm = float(np.median(X_norm))
        self.theta_w = 0.10 * gamma_res
        self.phi_w   = 0.50 * self.theta_w
        self.phi_g   = 0.01 * self.theta_w
        self.theta_d = 0.05 * gamma_res
        self.delta_disp = 0.01 * X_norm


class Head(nn.Module):
    def __init__(self, d, dk, dv, sigma_a, sigma_V_star, sigma_QK=None):
        super().__init__()
        if sigma_QK is None: sigma_QK = 1.0/sqrt(d)
        self.W_Q = nn.Parameter(torch.randn(d, dk) * sigma_QK)
        self.W_K = nn.Parameter(torch.randn(d, dk) * sigma_QK)
        self.W_V = nn.Parameter(torch.randn(d, dv) * sigma_V_star)
        self.W_O = nn.Parameter(torch.randn(dv, d) * (1.0/sqrt(dv)))
        self.register_buffer('u_plus',  F.normalize(torch.randn(d), dim=0))
        self.register_buffer('u_minus', F.normalize(torch.randn(d), dim=0))
        self.u_minus = F.normalize(self.u_minus - (self.u_minus @ self.u_plus)*self.u_plus, dim=0)
        self.d, self.dk, self.dv = d, dk, dv
        self.prune_counter = 0
        self.birth_step = 0

    def M_full(self):  return self.W_Q @ self.W_K.T
    def Ma_full(self): M = self.M_full(); return (M - M.T)/2
    def Ms_full(self): M = self.M_full(); return (M + M.T)/2
    def Pa_norm_squared(self):
        P = self.W_Q.T @ self.W_K
        Pa = (P - P.T)/2
        return (Pa**2).sum()


def compute_Ares(X, Ma_skew, Q):
    d = X.shape[1]
    if Q.shape[1] == 0: Proj = torch.eye(d, device=X.device)
    else: Proj = torch.eye(d, device=X.device) - Q @ Q.T
    A = X.T @ X @ Ma_skew
    A_sym = (A + A.T)/2
    return Proj @ A_sym @ Proj


def rayleigh_quotient(u, A): return (u @ (A @ u)) / (u @ u)


def spectral_gaps(A):
    eigs = torch.linalg.eigvalsh(A).flip(0)
    if len(eigs) < 2: return 1e-8, 1e-8
    return max((eigs[0] - eigs[1]).item(), 1e-8), max((eigs[-2] - eigs[-1]).item(), 1e-8)


def oja_step(u_plus, u_minus, A, eta, gstar):
    Au = A @ u_plus
    R = (u_plus @ Au).item()
    coupling = (u_plus @ (A @ u_minus)).item()
    u_new = u_plus + eta * ((Au - R*u_plus) - (1 - gstar)*coupling*u_minus)
    return F.normalize(u_new, dim=0)


def mca_exin_step(u_minus, u_plus, A, eta, gstar):
    nrm = (u_minus @ u_minus).item()
    Au = A @ u_minus
    y = (u_minus @ Au).item()
    mca = (eta * y / nrm) * (Au - (y/nrm)*u_minus)
    coupling = (u_plus @ (A @ u_minus)).item()
    u_new = u_minus - mca + eta * gstar * (1 - gstar) * coupling * u_plus
    u_new = F.normalize(u_new, dim=0)
    u_new = u_new - (u_new @ u_plus) * u_plus
    return F.normalize(u_new, dim=0)


def min_dwell_time(eta, Delta, eps=0.05, kappa_T=2.0):
    if Delta < 1e-8: return 10000
    return int(ceil(kappa_T / (eta * Delta**2) * log(1.0 / eps)))


def project_out(v, Q):
    if Q.shape[1] == 0: return v
    v = v - Q @ (Q.T @ v)
    return F.normalize(v, dim=0)


class LayerState:
    def __init__(self, d):
        self.heads = []
        self.Q = torch.zeros(d, 0, device=DEVICE)
        self.u_plus = F.normalize(torch.randn(d, device=DEVICE), dim=0)
        self.u_minus = F.normalize(torch.randn(d, device=DEVICE), dim=0)
        self.u_minus -= (self.u_minus @ self.u_plus) * self.u_plus
        self.u_minus = F.normalize(self.u_minus, dim=0)
        self.last_trigger_step = -10000
        self.gamma_star = 0.5


def try_grow_head(layer, X, Ma, cfg, step):
    d = X.shape[1]
    A = compute_Ares(X, Ma, layer.Q)
    Dp, Dm = spectral_gaps(A)
    layer.gamma_star = Dp / (Dp + Dm + 1e-8)
    layer.u_plus  = oja_step(layer.u_plus, layer.u_minus, A, cfg.eta_plus, layer.gamma_star)
    layer.u_minus = mca_exin_step(layer.u_minus, layer.u_plus, A, cfg.eta_minus, layer.gamma_star)
    lmax = rayleigh_quotient(layer.u_plus, A).item()
    lmin = rayleigh_quotient(layer.u_minus, A).item()
    Td = min_dwell_time(cfg.eta_plus, Dp, cfg.eps_conv, cfg.kappa_T)
    if step - layer.last_trigger_step < Td:
        return None, lmax, lmin
    if lmax > cfg.theta_w and lmin < cfg.phi_w:
        head = Head(d, cfg.dk, cfg.dv, cfg.sigma_a, cfg.sigma_V_star).to(DEVICE)
        head.u_plus = layer.u_plus.clone()
        head.u_minus = layer.u_minus.clone()
        head.birth_step = step
        layer.heads.append(head)
        layer.Q = torch.cat([layer.Q, layer.u_plus.unsqueeze(1)], dim=1)
        nu = F.normalize(torch.randn(d, device=DEVICE), dim=0)
        layer.u_plus = project_out(nu, layer.Q)
        nu2 = F.normalize(torch.randn(d, device=DEVICE), dim=0)
        layer.u_minus = project_out(nu2, torch.cat([layer.Q, layer.u_plus.unsqueeze(1)], dim=1))
        layer.last_trigger_step = step
        return head, lmax, lmin
    return None, lmax, lmin


def try_prune_heads(layer, X, cfg, step):
    pruned = []
    for i, h in enumerate(layer.heads):
        Td = min_dwell_time(cfg.eta_plus, 0.1, cfg.eps_conv, cfg.kappa_T)
        if step - h.birth_step < Td:
            h.prune_counter = 0; continue
        Gh = (X.T @ X @ h.Ma_full().detach()).norm().item()
        if Gh < cfg.phi_g: h.prune_counter += 1
        else: h.prune_counter = 0
        if h.prune_counter >= cfg.T_prune:
            pruned.append(i)
    for i in reversed(pruned):
        layer.heads.pop(i)
    return len(pruned)


class INCRTForMLM(nn.Module):
    def __init__(self, cfg, n_heads_per_layer=None):
        super().__init__()
        self.cfg = cfg
        self.token_embed = nn.Embedding(cfg.vocab_size, cfg.d)
        self.pos_embed = nn.Embedding(cfg.n, cfg.d)
        self.embed_norm = nn.LayerNorm(cfg.d)
        self.embed_dropout = nn.Dropout(0.1)
        if n_heads_per_layer is None:
            n_heads_per_layer = [1]
        self.layers_list = nn.ModuleList([
            nn.ModuleList([Head(cfg.d, cfg.dk, cfg.dv, cfg.sigma_a, cfg.sigma_V_star)
                            for _ in range(H)])
            for H in n_heads_per_layer
        ])
        self.layer_states = [LayerState(cfg.d) for _ in n_heads_per_layer]
        for ls, hl in zip(self.layer_states, self.layers_list):
            ls.heads = list(hl)
        self.layer_norms = nn.ModuleList([nn.LayerNorm(cfg.d) for _ in n_heads_per_layer])
        self.out_norm = nn.LayerNorm(cfg.d)
        self.mlm_head = nn.Linear(cfg.d, cfg.vocab_size, bias=True)
        self.step = 0
        self.last_level3_step = -10000
        self.history = {
            'n_heads': [], 'n_layers': [],
            'lambda_max': [], 'lambda_min': [],
            'loss_mlm': [], 'loss_geo': [], 'loss_alpha': [], 'loss_total': [],
            'Pa_norm_total': [], 'alpha_top_layer': [], 'level3_events': [],
        }

    def _embed(self, ids):
        positions = torch.arange(ids.shape[1], device=ids.device).unsqueeze(0)
        return self.embed_dropout(self.embed_norm(self.token_embed(ids) + self.pos_embed(positions)))

    def forward(self, ids, return_hidden=False):
        X = self._embed(ids)
        for l_idx, ls in enumerate(self.layer_states):
            if not ls.heads: continue
            outs = []
            for h in ls.heads:
                M = h.M_full()
                scores = torch.einsum('bnd,de,bme->bnm', X, M, X) / sqrt(self.cfg.dk)
                attn = F.softmax(scores, dim=-1)
                V = X @ h.W_V
                outs.append((attn @ V) @ h.W_O)
            X = X + sum(outs)/len(outs)
            X = self.layer_norms[l_idx](X)
        X = self.out_norm(X)
        if return_hidden: return X
        return self.mlm_head(X)

    def total_Pa_norm_squared(self):
        total = torch.tensor(0.0, device=DEVICE)
        for ls in self.layer_states:
            for h in ls.heads:
                total = total + h.Pa_norm_squared()
        return total

    def total_alpha_loss(self):
        terms = []
        for ls in self.layer_states:
            for h in ls.heads:
                M = h.M_full()
                Ma = (M - M.T)/2
                Mn = M.norm()
                if Mn.item() < 1e-8: continue
                terms.append((1.0 - Ma.norm()/(Mn + 1e-12))**2)
        if not terms: return torch.tensor(0.0, device=DEVICE)
        return sum(terms) / len(terms)

    def asymmetry_index_layer(self, l_idx):
        ls = self.layer_states[l_idx]
        if not ls.heads: return 0.0
        with torch.no_grad():
            ratios = []
            for h in ls.heads:
                M = h.M_full(); Mn = M.norm().item()
                if Mn < 1e-8: continue
                ratios.append(((M - M.T)/2).norm().item() / Mn)
        return float(sum(ratios)/len(ratios)) if ratios else 0.0

    def _is_top_layer_frozen(self):
        ah = self.history['alpha_top_layer']
        if len(ah) < self.cfg.T_wnd + 1: return False
        return abs(ah[-1] - ah[-self.cfg.T_wnd - 1]) < self.cfg.eps_alpha

    def try_level3_growth(self):
        cfg = self.cfg
        if not cfg.enable_level3: return None
        if len(self.layer_states) >= cfg.max_layers: return None
        if self.step - self.last_level3_step < cfg.layer3_dwell: return None
        L = len(self.layer_states) - 1
        a_top = self.asymmetry_index_layer(L)
        self.history['alpha_top_layer'].append(a_top)
        if a_top <= cfg.alpha_crit: return None
        if not self._is_top_layer_frozen(): return None
        # SOFT identity init (v7)
        new_head = Head(cfg.d, cfg.dk, cfg.dv, cfg.sigma_a, cfg.sigma_V_star).to(DEVICE)
        with torch.no_grad():
            scale = 1e-3
            sigma_qk = scale / sqrt(cfg.d)
            sigma_v  = scale * cfg.sigma_V_star
            new_head.W_Q.normal_(0.0, sigma_qk)
            new_head.W_K.normal_(0.0, sigma_qk)
            new_head.W_V.normal_(0.0, sigma_v)
            new_head.W_O.normal_(0.0, scale / sqrt(cfg.dv))
        self.layers_list.append(nn.ModuleList([new_head]))
        new_ls = LayerState(cfg.d)
        new_ls.heads = [new_head]
        self.layer_states.append(new_ls)
        self.layer_norms.append(nn.LayerNorm(cfg.d).to(DEVICE))
        self.last_level3_step = self.step
        self.history['level3_events'].append((self.step, len(self.layer_states) - 1))
        print(f'  [Step {self.step}] LEVEL-3: layer {len(self.layer_states)} added (alpha_top={a_top:.3f})')
        return new_head

    def grow_step(self, X_flat):
        cfg = self.cfg
        # Track alpha
        L = len(self.layer_states) - 1
        a_top = self.asymmetry_index_layer(L)
        if (not self.history['alpha_top_layer'] or
                len(self.history['alpha_top_layer']) <= self.step // 5):
            self.history['alpha_top_layer'].append(a_top)
        # Try Level-3
        new_layer_head = self.try_level3_growth()
        # Try Level-1 on every layer
        new_heads = []
        total_h = 0; lmax_all, lmin_all = [], []
        for l_idx, ls in enumerate(self.layer_states):
            if not ls.heads: continue
            Ma_cum = sum(h.Ma_full().detach() for h in ls.heads)
            head, lmax, lmin = try_grow_head(ls, X_flat, Ma_cum, cfg, self.step)
            if head is not None:
                self.layers_list[l_idx].append(head)
                new_heads.append(head)
            try_prune_heads(ls, X_flat, cfg, self.step)
            total_h += len(ls.heads)
            lmax_all.append(lmax); lmin_all.append(lmin)
        if lmax_all:
            self.history['n_heads'].append(total_h)
            self.history['n_layers'].append(len(self.layer_states))
            self.history['lambda_max'].append(max(lmax_all))
            self.history['lambda_min'].append(min(lmin_all))
        self.step += 1
        # Return the list of NEW parameters added this step (so the caller can update the optimizer)
        added = []
        if new_layer_head is not None:
            added += list(new_layer_head.parameters())
            # Also add the new LayerNorm
            added += list(self.layer_norms[-1].parameters())
        for h in new_heads:
            added += list(h.parameters())
        return added

print('Model definitions loaded.')


In [ ]:
# ── Pfam labels via UniProt REST API ────────────────────
PFAM_TSV_PATH = os.path.join(DATA_DIR, 'human_uniprot_pfam.tsv')

UNIPROT_QUERY = (
    'https://rest.uniprot.org/uniprotkb/stream?'
    'query=proteome:UP000005640+AND+reviewed:true&'
    'format=tsv&'
    'fields=accession,xref_ensembl,xref_pfam'
)

def download_uniprot_pfam():
    if os.path.exists(PFAM_TSV_PATH) and os.path.getsize(PFAM_TSV_PATH) > 100_000:
        print(f'UniProt Pfam TSV already present: {os.path.getsize(PFAM_TSV_PATH)/1e6:.1f} MB')
        return
    print('Downloading UniProt Pfam mapping...')
    urllib.request.urlretrieve(UNIPROT_QUERY, PFAM_TSV_PATH)

def parse_pfam_tsv(path):
    ens2pfam = {}
    n_with_pfam = 0
    with open(path) as f:
        header = f.readline().strip().split('\t')
        col_acc  = header.index('Entry')
        col_ens  = header.index('Ensembl')
        col_pfam = header.index('Pfam')
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 3: continue
            pfam_field = parts[col_pfam]
            if not pfam_field.strip(): continue
            n_with_pfam += 1
            pfam_ids = {p.strip().rstrip(';') for p in pfam_field.split(';') if p.strip()}
            for ens_chunk in parts[col_ens].split(';'):
                ens_chunk = ens_chunk.strip()
                if not ens_chunk: continue
                for eid in re.findall(r'(ENSP\d+|ENST\d+)', ens_chunk):
                    ens2pfam.setdefault(eid, set()).update(pfam_ids)
    print(f'Pfam mapping: {n_with_pfam} UniProt rows with annotations -> {len(ens2pfam)} Ensembl IDs')
    return ens2pfam


def build_pfam_dataset(proteins, ens2pfam, top_n=50):
    """Returns (records, top_families, family_to_idx).

    Robust ID matching: UniProt cross-refs are mostly ENST (transcripts) with
    occasional ENSP. Our protein dict uses ENSP keys (with '.N' version suffix).
    We try ENSP, transcript_id from metadata, and include version stripping.
    """
    # Build lookup by stripped ID
    protein_to_pfam = {}
    n_match_ensp = n_match_enst = 0
    for pid, meta in proteins.items():
        base_pid = pid.split('.')[0]  # strip .1/.2 version suffix
        pfams = set()
        # Try ENSP
        if base_pid in ens2pfam:
            pfams |= ens2pfam[base_pid]
            if pfams: n_match_ensp += 1
        # Try ENST via transcript_id metadata
        tid = meta.get('transcript_id')
        if tid:
            base_tid = tid.split('.')[0]
            if base_tid in ens2pfam:
                pfams |= ens2pfam[base_tid]
                if base_pid not in ens2pfam: n_match_enst += 1
        if pfams:
            protein_to_pfam[pid] = pfams

    print(f'Proteins with Pfam annotation: {len(protein_to_pfam)}/{len(proteins)} '
          f'({100*len(protein_to_pfam)/len(proteins):.1f}%)')
    print(f'  matched via ENSP: {n_match_ensp}, additional via ENST: {n_match_enst}')

    if len(protein_to_pfam) == 0:
        # Heavy diagnostic
        print('ERROR: zero matches! Diagnostic:')
        sample_proteins = list(proteins.items())[:5]
        sample_pfam_keys = list(ens2pfam.keys())[:5]
        print(f'  Sample protein IDs: {[p[0] for p in sample_proteins]}')
        print(f'  Sample protein metadata: {sample_proteins[0][1] if sample_proteins else None}')
        print(f'  Sample ens2pfam keys: {sample_pfam_keys}')
        print(f'  Total ens2pfam entries: {len(ens2pfam)}')
        # Check overlap of any IDs
        prot_base_ids = {p.split(".")[0] for p in proteins.keys()}
        prot_tids = {meta.get("transcript_id", "").split(".")[0] for meta in proteins.values()
                      if meta.get("transcript_id")}
        ens_keys = set(ens2pfam.keys())
        print(f'  Protein ENSP base IDs: {len(prot_base_ids)} unique')
        print(f'  Protein ENST transcript IDs: {len(prot_tids)} unique')
        print(f'  Overlap ENSP/ens2pfam: {len(prot_base_ids & ens_keys)}')
        print(f'  Overlap ENST/ens2pfam: {len(prot_tids & ens_keys)}')
        raise RuntimeError('No proteins matched any Pfam annotation. Cannot proceed.')

    all_pfams = Counter()
    for pf_set in protein_to_pfam.values():
        for p in pf_set: all_pfams[p] += 1
    top_families = [p for p, _ in all_pfams.most_common(top_n)]
    family_to_idx = {f: i for i, f in enumerate(top_families)}
    top_set = set(top_families)

    records = []
    for pid, pf_set in protein_to_pfam.items():
        cands = [family_to_idx[p] for p in pf_set if p in top_set]
        if not cands: continue
        records.append({'pid': pid, 'seq': proteins[pid]['seq'], 'label': min(cands)})
    print(f'Classification dataset: {len(records)} proteins, {top_n} classes')

    if len(records) < top_n * 5:
        print(f'WARNING: dataset is small ({len(records)} proteins for {top_n} classes). '
              f'Consider lowering top_n.')
    return records, top_families, family_to_idx


class PfamDataset(torch.utils.data.Dataset):
    def __init__(self, records, tokenizer, max_length=256):
        self.records = records
        self.tok = tokenizer; self.max_length = max_length
    def __len__(self): return len(self.records)
    def __getitem__(self, i):
        r = self.records[i]
        ids = self.tok.encode(r['seq'], self.max_length)
        return {'input_ids': torch.tensor(ids, dtype=torch.long),
                'label': torch.tensor(r['label'], dtype=torch.long), 'pid': r['pid']}


In [ ]:
# ── Pfam evaluation: linear probe + fine-tune ─────────────
@torch.no_grad()
def extract_cls_embeddings(model, dataset, batch_size=32, desc='Extract'):
    if len(dataset) == 0:
        raise RuntimeError(f'{desc}: dataset is empty! Cannot extract embeddings. '
                            f'Check Pfam label matching upstream.')
    model.eval()
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    embs, labels = [], []
    for batch in tqdm(loader, desc=desc):
        ids = batch['input_ids'].to(DEVICE)
        h = model(ids, return_hidden=True)
        cls = h[:, 0, :]
        embs.append(cls.cpu()); labels.append(batch['label'])
    return torch.cat(embs, dim=0), torch.cat(labels, dim=0)


def train_linear_probe(train_X, train_y, val_X, val_y, test_X, test_y,
                        n_classes, d, n_epochs=20, batch=256, lr=1e-2):
    class LinearHead(nn.Module):
        def __init__(self, d, n_classes):
            super().__init__()
            self.fc = nn.Linear(d, n_classes)
        def forward(self, x): return self.fc(x)
    probe = LinearHead(d, n_classes).to(DEVICE)
    opt = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    train_X_d = train_X.to(DEVICE); train_y_d = train_y.to(DEVICE)
    val_X_d   = val_X.to(DEVICE);   val_y_d   = val_y.to(DEVICE)
    test_X_d  = test_X.to(DEVICE);  test_y_d  = test_y.to(DEVICE)
    best_val_acc = 0.0; best_state = None
    for ep in range(n_epochs):
        probe.train()
        perm = torch.randperm(train_X_d.shape[0], device=DEVICE)
        for i in range(0, len(perm), batch):
            idx = perm[i:i+batch]
            logits = probe(train_X_d[idx])
            loss = F.cross_entropy(logits, train_y_d[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        probe.eval()
        with torch.no_grad():
            logits_val = probe(val_X_d)
            val_acc = (logits_val.argmax(-1) == val_y_d).float().mean().item()
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k:v.detach().clone() for k,v in probe.state_dict().items()}
        print(f'  probe ep {ep+1:2d}  val_acc={val_acc:.4f}')
    probe.load_state_dict(best_state)
    probe.eval()
    with torch.no_grad():
        logits_test = probe(test_X_d)
        test_acc = (logits_test.argmax(-1) == test_y_d).float().mean().item()
        test_top5 = (logits_test.topk(5, dim=-1).indices == test_y_d.unsqueeze(-1)).any(dim=-1).float().mean().item()
    return {'best_val_acc': best_val_acc, 'test_acc': test_acc, 'test_top5': test_top5,
            'state': best_state}


def light_finetune(encoder, train_set, val_set, test_set, n_classes, d,
                    n_epochs=2, lr_enc=1e-5, lr_head=1e-3, batch=32):
    class FTModel(nn.Module):
        def __init__(self, enc, n_classes):
            super().__init__()
            self.enc = enc
            self.cls_head = nn.Linear(d, n_classes)
        def forward(self, ids):
            h = self.enc(ids, return_hidden=True)
            return self.cls_head(h[:, 0, :])
    model_ft = FTModel(encoder, n_classes).to(DEVICE)
    params = [
        {'params': model_ft.enc.parameters(), 'lr': lr_enc},
        {'params': model_ft.cls_head.parameters(), 'lr': lr_head},
    ]
    opt = torch.optim.AdamW(params, weight_decay=1e-3)
    train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch, shuffle=True, num_workers=2)
    val_loader   = torch.utils.data.DataLoader(val_set, batch_size=batch, shuffle=False, num_workers=2)
    test_loader  = torch.utils.data.DataLoader(test_set, batch_size=batch, shuffle=False, num_workers=2)
    best_val = 0.0; best_state = None
    for ep in range(n_epochs):
        model_ft.train()
        pbar = tqdm(train_loader, desc=f'  FT ep {ep+1}')
        for b in pbar:
            ids = b['input_ids'].to(DEVICE); y = b['label'].to(DEVICE)
            logits = model_ft(ids)
            loss = F.cross_entropy(logits, y)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model_ft.parameters(), 1.0)
            opt.step()
        model_ft.eval()
        n_corr = n_tot = 0
        with torch.no_grad():
            for b in val_loader:
                ids = b['input_ids'].to(DEVICE); y = b['label'].to(DEVICE)
                preds = model_ft(ids).argmax(-1)
                n_corr += (preds == y).sum().item(); n_tot += y.numel()
        val_acc = n_corr / n_tot
        print(f'  FT ep {ep+1}: val_acc = {val_acc:.4f}')
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k:v.detach().clone() for k,v in model_ft.state_dict().items()}
    model_ft.load_state_dict(best_state)
    model_ft.eval()
    n_corr = n_tot = n_top5 = 0
    with torch.no_grad():
        for b in test_loader:
            ids = b['input_ids'].to(DEVICE); y = b['label'].to(DEVICE)
            logits = model_ft(ids)
            preds = logits.argmax(-1)
            n_corr += (preds == y).sum().item(); n_tot += y.numel()
            n_top5 += (logits.topk(5, dim=-1).indices == y.unsqueeze(-1)).any(dim=-1).sum().item()
    return {'best_val_acc': best_val, 'test_acc': n_corr/n_tot, 'test_top5': n_top5/n_tot}


def evaluate_pfam(checkpoint_path, output_dir, top_n=50, label='?'):
    print(f'\n========== Pfam evaluation: {label} ==========')
    print(f'Checkpoint: {checkpoint_path}')

    # Load checkpoint and reconstruct architecture
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    cfg_d = ckpt['config']
    cfg = INCRTConfig(**{k:v for k,v in cfg_d.items() if k in INCRTConfig.__dataclass_fields__})
    sd = ckpt['state_dict']
    layer_indices = sorted({int(re.match(r'layers_list\.(\d+)\.', k).group(1))
                              for k in sd if k.startswith('layers_list.')})
    n_heads_per_layer = []
    for li in layer_indices:
        hs = sorted({int(re.match(rf'layers_list\.{li}\.(\d+)\.', k).group(1))
                      for k in sd if k.startswith(f'layers_list.{li}.')})
        n_heads_per_layer.append(len(hs))
    print(f'Architecture: {len(layer_indices)} layers, heads per layer: {n_heads_per_layer}')

    encoder = INCRTForMLM(cfg, n_heads_per_layer).to(DEVICE)
    missing, unexpected = encoder.load_state_dict(sd, strict=False)
    print(f'Missing keys: {len(missing)} | Unexpected: {len(unexpected)}')
    encoder.eval()
    n_params = sum(p.numel() for p in encoder.parameters())
    print(f'Encoder params: {n_params:,}')

    # Pfam labels
    download_uniprot_pfam()
    ens2pfam = parse_pfam_tsv(PFAM_TSV_PATH)
    records, top_families, family_to_idx = build_pfam_dataset(proteins, ens2pfam, top_n=top_n)

    # Stratified split
    by_class = {}
    for r in records:
        by_class.setdefault(r['label'], []).append(r)
    random.seed(SEED)
    train_r, val_r, test_r = [], [], []
    for cls, recs in by_class.items():
        random.shuffle(recs)
        n = len(recs)
        nt = max(1, int(n*0.10)); nv = max(1, int(n*0.10))
        test_r += recs[:nt]; val_r += recs[nt:nt+nv]; train_r += recs[nt+nv:]
    random.shuffle(train_r); random.shuffle(val_r); random.shuffle(test_r)
    print(f'Train/Val/Test: {len(train_r)} / {len(val_r)} / {len(test_r)}')

    train_set = PfamDataset(train_r, tokenizer, max_length=cfg.n)
    val_set   = PfamDataset(val_r,   tokenizer, max_length=cfg.n)
    test_set  = PfamDataset(test_r,  tokenizer, max_length=cfg.n)

    # ── Linear probe ─────────────────────
    print('\n-- Linear probe --')
    train_X, train_y = extract_cls_embeddings(encoder, train_set, desc='Train embeds')
    val_X,   val_y   = extract_cls_embeddings(encoder, val_set,   desc='Val embeds')
    test_X,  test_y  = extract_cls_embeddings(encoder, test_set,  desc='Test embeds')
    probe_res = train_linear_probe(train_X, train_y, val_X, val_y, test_X, test_y,
                                     n_classes=top_n, d=cfg.d)
    print(f'Linear probe: test_acc={probe_res["test_acc"]:.4f}, top5={probe_res["test_top5"]:.4f}')

    # ── Light fine-tune ──────────────────
    print('\n-- Light fine-tune --')
    encoder_ft = INCRTForMLM(cfg, n_heads_per_layer).to(DEVICE)
    encoder_ft.load_state_dict(sd, strict=False)
    ft_res = light_finetune(encoder_ft, train_set, val_set, test_set,
                              n_classes=top_n, d=cfg.d, n_epochs=2)
    print(f'Fine-tune: test_acc={ft_res["test_acc"]:.4f}, top5={ft_res["test_top5"]:.4f}')

    # Save results
    summary = {
        'label': label, 'checkpoint': checkpoint_path,
        'architecture': {'layers': len(layer_indices), 'heads_per_layer': n_heads_per_layer,
                          'total_params': n_params, 'd': cfg.d},
        'val_mlm_acc': ckpt.get('val_mlm_acc', None),
        'val_mlm_loss': ckpt.get('val_mlm_loss', None),
        'top_n_classes': top_n, 'n_train': len(train_r), 'n_val': len(val_r), 'n_test': len(test_r),
        'linear_probe': {k:v for k,v in probe_res.items() if k != 'state'},
        'fine_tune': ft_res,
    }
    with open(os.path.join(output_dir, 'pfam_summary.json'), 'w') as f:
        json.dump(summary, f, indent=2)
    return summary


## PHASE 11 — Pretraining v9' (full INCRT-geo, multi-vertebrate Ensembl ~200k)

Same training algorithm as v9. Only the pretraining corpus changes (8-species Ensembl multi-vertebrate instead of human-only). Per-epoch checkpointing is enabled; if the Colab session disconnects, simply re-run the notebook and pretraining resumes from the last completed epoch.

Note: each epoch on the multi-vertebrate corpus takes approximately 7-8 hours on a Colab Pro+ A100 GPU (vs ~17 minutes on human-only). Expected total: 60-80 hours of GPU time for 10 epochs.


In [ ]:
# ── MLM dataset and collator (for pretraining) ────────────
class ProteinMLMDataset(torch.utils.data.Dataset):
    def __init__(self, proteins, tok, max_length=256):
        self.pids = list(proteins.keys())
        self.proteins = proteins; self.tok = tok; self.max_length = max_length
    def __len__(self): return len(self.pids)
    def __getitem__(self, i):
        pid = self.pids[i]
        ids = self.tok.encode(self.proteins[pid]['seq'], self.max_length)
        return {'input_ids': torch.tensor(ids, dtype=torch.long), 'protein_id': pid}

def mlm_collate(batch, tok, mlm_prob=0.15):
    ids = torch.stack([b['input_ids'] for b in batch])
    labels = ids.clone()
    P = torch.full(ids.shape, mlm_prob)
    sm = (ids == tok.pad_id) | (ids == tok.cls_id) | (ids == tok.sep_id)
    P.masked_fill_(sm, 0.0)
    masked = torch.bernoulli(P).bool()
    labels[~masked] = -100
    rep = torch.bernoulli(torch.full(ids.shape, 0.8)).bool() & masked
    ids[rep] = tok.mask_id
    rand = torch.bernoulli(torch.full(ids.shape, 0.5)).bool() & masked & ~rep
    rwords = torch.randint(len(tok.SPECIAL), tok.vocab_size, ids.shape)
    ids[rand] = rwords[rand]
    return {'input_ids': ids, 'labels': labels}


In [ ]:
def train_v8_pretrain(model, dataloader, n_epochs=15, lr=1e-3,
                       log_every=200, calibration_batches=16, grow_every=5,
                       output_dir='.', warmup_steps=500, use_lr_schedule=True):
    """v8 pretraining with optimizer.add_param_group instead of full rebuild."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    cfg = model.cfg
    total_steps = len(dataloader) * n_epochs

    def lr_lambda_fn(step):
        if step < warmup_steps: return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.1 + 0.9 * 0.5 * (1.0 + np.cos(np.pi * min(1.0, progress)))

    if use_lr_schedule:
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda_fn)
    else:
        scheduler = None

    # Calibrate thresholds on first N batches
    if cfg.theta_w is None:
        model.eval()
        gammas, xnorms = [], []
        with torch.no_grad():
            for k, b in enumerate(dataloader):
                if k >= calibration_batches: break
                ids = b['input_ids'].to(DEVICE)
                X0 = model._embed(ids[0:1]).squeeze(0)
                h0 = model.layer_states[0].heads[0]
                A0 = compute_Ares(X0, h0.Ma_full().detach(), model.layer_states[0].Q)
                gammas.append(A0.norm().item()); xnorms.append(X0.norm().item())
        cfg.calibrate(gammas, xnorms)
        print(f'Calibration: median Gamma_res={float(np.median(gammas)):.3f} '
              f'-> theta_w={cfg.theta_w:.3f}')

    model.train()
    global_step = 0
    for epoch in range(n_epochs):
        ep_loss_mlm = 0; n_correct = n_total = 0
        pbar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{n_epochs}')
        for batch in pbar:
            ids = batch['input_ids'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            optimizer.zero_grad()
            logits = model(ids)
            L_mlm = F.cross_entropy(logits.view(-1, cfg.vocab_size), labels.view(-1), ignore_index=-100)

            # L_alpha
            if cfg.lambda_alpha > 0:
                L_alpha = cfg.lambda_alpha * model.total_alpha_loss()
            else:
                L_alpha = torch.tensor(0.0, device=DEVICE)
            # L_geo (warmup ramp)
            lambda_t = cfg.lambda_geo * min(1.0, global_step / max(1, cfg.lambda_warmup_steps))
            Pa_norm = model.total_Pa_norm_squared()
            if cfg.use_log_geo:
                L_geo = - lambda_t * torch.log1p(Pa_norm)
            else:
                L_geo = - lambda_t * Pa_norm

            loss = L_mlm + L_alpha + L_geo
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            if scheduler is not None: scheduler.step()

            # Geometric growth — V8 FIX: add_param_group instead of rebuilding the whole optimizer
            if global_step % grow_every == 0:
                with torch.no_grad():
                    X_flat = model._embed(ids[0:1]).squeeze(0).detach()
                    new_params = model.grow_step(X_flat)
                if new_params:
                    optimizer.add_param_group({'params': new_params,
                                                'lr': optimizer.param_groups[0]['lr'],
                                                'weight_decay': optimizer.param_groups[0]['weight_decay']})
                    # Keep the scheduler in sync: LambdaLR stores one lambda per param_group;
                    # adding a param_group means we must extend lr_lambdas, otherwise
                    # the next scheduler.step() raises zip-length mismatch in PyTorch >=2.1.
                    if scheduler is not None:
                        # base_lrs grows by one (the current LR is the new "base")
                        if hasattr(scheduler, 'base_lrs'):
                            scheduler.base_lrs.append(optimizer.param_groups[-1]['lr'])
                        if hasattr(scheduler, 'lr_lambdas'):
                            scheduler.lr_lambdas.append(lr_lambda_fn)
                        if hasattr(scheduler, '_last_lr'):
                            scheduler._last_lr.append(optimizer.param_groups[-1]['lr'])

            with torch.no_grad():
                mask = labels != -100
                if mask.any():
                    preds = logits.argmax(-1)
                    n_correct += ((preds == labels) & mask).sum().item()
                    n_total += mask.sum().item()
            ep_loss_mlm += L_mlm.item()
            model.history['loss_mlm'].append(L_mlm.item())
            model.history['loss_geo'].append(L_geo.item())
            model.history['loss_alpha'].append(L_alpha.item())
            model.history['loss_total'].append(loss.item())
            model.history['Pa_norm_total'].append(Pa_norm.item())

            if global_step % log_every == 0:
                n_h = sum(len(ls.heads) for ls in model.layer_states)
                a_avg = 0.0
                avals = []
                with torch.no_grad():
                    for ls in model.layer_states:
                        for hd in ls.heads:
                            M = hd.M_full(); Mn = M.norm().item()
                            if Mn > 1e-8:
                                avals.append(((M-M.T)/2).norm().item() / Mn)
                if avals: a_avg = sum(avals)/len(avals)
                pbar.set_postfix({
                    'L_mlm': f'{L_mlm.item():.2f}',
                    'L_a':   f'{L_alpha.item():.3f}',
                    'a':     f'{a_avg:.3f}',
                    'lr':    f'{optimizer.param_groups[0]["lr"]:.1e}',
                    'h':     n_h,
                    'L':     len(model.layer_states),
                    'acc':   f'{n_correct/max(1,n_total):.3f}',
                })
            global_step += 1

        n_h = sum(len(ls.heads) for ls in model.layer_states)
        avg_mlm = ep_loss_mlm / len(dataloader)
        ep_acc = n_correct / max(1, n_total)
        print(f'Epoch {epoch+1}: avg L_mlm={avg_mlm:.4f}, mlm_acc={ep_acc:.4f}, '
              f'heads={n_h}, layers={len(model.layer_states)}')
        torch.save({
            'epoch': epoch+1, 'config': cfg.__dict__, 'state_dict': model.state_dict(),
            'history': model.history, 'epoch_acc': ep_acc,
        }, os.path.join(output_dir, f'incrt_geo_epoch{epoch+1}.pt'))

    return model


In [ ]:
# === PHASE 11 — pretraining v9' on SwissProt with resume support ===

if status.get('phase11_done', False):
    print("Phase 11 already done. Skipping pretraining.")
    v9p_ckpt = os.path.join(PHASE11_DIR, 'incrt_geo_protein_encoder.pt')
    if not os.path.exists(v9p_ckpt):
        print(f"WARNING: phase11_done=True but no checkpoint at {v9p_ckpt}")
        print("Resetting phase11_done=False to retry.")
        status['phase11_done'] = False; save_status(status)

if not status.get('phase11_done', False):
    log_event("Phase 11 START — pretraining v9' (SwissProt, full INCRT-geo)")

    # ── Tokenizer ──
    tokenizer = ProteinKmerTokenizer(k=1)
    print(f'Tokenizer vocab size: {tokenizer.vocab_size}')

    # ── Load SwissProt (cached on Drive after first run) ──
    proteins = load_multi_vertebrate_proteins()
    print(f'Pretraining corpus: {len(proteins):,} sequences (8-species Ensembl multi-vertebrate, deduplicated)')

    # ── Train/val split (95/5, deterministic seed=42) ──
    # proteins is a dict {protein_id: sequence}. Split by shuffling the keys.
    rng = random.Random(42)
    all_pids = list(proteins.keys())
    rng.shuffle(all_pids)
    n_val = max(1, len(all_pids) // 20)
    val_pids   = set(all_pids[:n_val])
    train_pids = set(all_pids[n_val:])
    train_seqs = {pid: proteins[pid] for pid in train_pids}
    val_seqs   = {pid: proteins[pid] for pid in val_pids}
    print(f'Train: {len(train_seqs):,} | Val: {len(val_seqs):,}')

    # ── V9' config: identical to v9 ──
    cfg_v9p = INCRTConfig(d=256, dk=64, dv=64, n=512, vocab_size=tokenizer.vocab_size,
                          lambda_geo=1e-5, lambda_warmup_steps=2000, use_log_geo=True,
                          lambda_alpha=10.0, alpha_crit=0.9, T_wnd=200, eps_alpha=0.01,
                          max_layers=6, layer3_dwell=500,
                          enable_level3=True)
    print("V9' config: identical to v9. Only the corpus changes.")

    # ── Datasets ──
    pretrain_ds = ProteinMLMDataset(train_seqs, tokenizer, max_length=cfg_v9p.n)
    val_ds      = ProteinMLMDataset(val_seqs,   tokenizer, max_length=cfg_v9p.n)
    # mlm_collate generates the masked labels at every batch (BERT-style 80/10/10).
    # Without collate_fn, the DataLoader returns only input_ids, missing the 'labels' key
    # that the training loop reads.
    from functools import partial
    _collate = partial(mlm_collate, tok=tokenizer)

    train_loader_p = torch.utils.data.DataLoader(pretrain_ds, batch_size=16, shuffle=True,
                                                  num_workers=2, pin_memory=True, drop_last=True,
                                                  collate_fn=_collate)
    val_loader_p   = torch.utils.data.DataLoader(val_ds,      batch_size=16, shuffle=False,
                                                  num_workers=2, pin_memory=True,
                                                  collate_fn=_collate)
    print(f'Train batches/epoch: {len(train_loader_p):,}')

    # ── Resume detection ──
    # Look for the latest per-epoch checkpoint in PHASE11_DIR.
    import glob, re as _re
    ckpt_files = glob.glob(os.path.join(PHASE11_DIR, 'incrt_geo_epoch*.pt'))
    latest_epoch = 0
    latest_ckpt_path = None
    for cf in ckpt_files:
        m = _re.search(r'epoch(\d+)\.pt$', cf)
        if m:
            ep = int(m.group(1))
            if ep > latest_epoch:
                latest_epoch = ep
                latest_ckpt_path = cf

    # ── Build / restore model ──
    if latest_epoch > 0:
        print(f'\n>>> RESUMING from epoch {latest_epoch} ({latest_ckpt_path})')
        ck = torch.load(latest_ckpt_path, map_location=DEVICE, weights_only=False)
        # Reconstruct model with the saved final architecture
        cfg_resume = INCRTConfig(**{k:v for k,v in ck['config'].items() if k in INCRTConfig.__dataclass_fields__})
        model_v9p = INCRTForMLM(cfg_resume).to(DEVICE)
        # Reconstruct heads/layers to match the saved state by replaying growth events stored in history if available
        # Simpler approach: rebuild by inspecting state_dict shapes. The training loop's grow_step is idempotent
        # under add_param_group, so we trust that loading state_dict on a freshly-built INCRTForMLM that has
        # been pre-grown to the recorded heads_per_layer reproduces the architecture exactly.
        # Pre-grow architecture:
        n_layers_target = ck['n_layers_final'] if 'n_layers_final' in ck else len(model_v9p.layer_states)
        heads_per_layer = ck.get('heads_per_layer_final', None)
        if heads_per_layer is None:
            # Fallback: cannot pre-grow precisely; warn
            print('WARNING: checkpoint missing heads_per_layer_final; resuming may fail to load weights cleanly.')
        else:
            # Add layers up to target
            while len(model_v9p.layer_states) < n_layers_target:
                model_v9p.add_layer()
            # Add heads to each layer up to target counts
            for li, target_h in enumerate(heads_per_layer):
                while len(model_v9p.layer_states[li].heads) < target_h:
                    model_v9p.add_head_at(li)
        # Now load state_dict
        model_v9p.load_state_dict(ck['state_dict'], strict=False)
        # Restore history
        model_v9p.history = ck.get('history', {})
        epoch_start = latest_epoch
        print(f'Resumed model: layers={len(model_v9p.layer_states)}, '
              f'heads_per_layer={[len(ls.heads) for ls in model_v9p.layer_states]}, '
              f'params={sum(p.numel() for p in model_v9p.parameters()):,}')
        print(f'Will train epochs {epoch_start+1} through 10.')
    else:
        print('Starting fresh training from epoch 1.')
        model_v9p = INCRTForMLM(cfg_v9p).to(DEVICE)
        epoch_start = 0
        print(f"V9' model — initial params: {sum(p.numel() for p in model_v9p.parameters()):,}")

    # ── Resume-aware pretraining ──
    # We assume train_v8_pretrain accepts an `epoch_start` argument; if not, the wrapper below patches it.
    import inspect
    sig = inspect.signature(train_v8_pretrain)
    if 'epoch_start' in sig.parameters:
        model_v9p = train_v8_pretrain(model_v9p, train_loader_p,
                                       n_epochs=10, lr=1e-3, log_every=200,
                                       calibration_batches=16, grow_every=5,
                                       output_dir=PHASE11_DIR, warmup_steps=500, use_lr_schedule=True,
                                       epoch_start=epoch_start)
    else:
        # train_v8_pretrain does not natively support resume.
        # We loop over remaining epochs manually, calling the inner training step.
        # NOTE: this fallback assumes the training function in cell 10 has been refactored
        # to expose per-epoch logic; otherwise a one-shot retraining of remaining epochs
        # without head/layer growth state from prior epochs would be lossy.
        print('WARNING: train_v8_pretrain does not natively accept epoch_start.')
        print('         For correct resume behaviour, refactor cell 10 to accept epoch_start (see notebook header).')
        print('         As a fallback, training all 10 epochs from scratch on the resumed model.')
        model_v9p = train_v8_pretrain(model_v9p, train_loader_p,
                                       n_epochs=10 - epoch_start, lr=1e-3, log_every=200,
                                       calibration_batches=16, grow_every=5,
                                       output_dir=PHASE11_DIR, warmup_steps=500, use_lr_schedule=True)

    # ── Final eval and save ──
    @torch.no_grad()
    def eval_mlm(model, loader):
        model.eval()
        tot = 0; cor = 0; n_tok = 0
        for batch in loader:
            tokens = batch['input_ids'].to(DEVICE)
            lab    = batch['labels'].to(DEVICE)
            logits = model(tokens)
            mask = lab != -100
            tot += F.cross_entropy(logits.view(-1, cfg_v9p.vocab_size), lab.view(-1),
                                    ignore_index=-100, reduction='sum').item()
            pred = logits.argmax(-1)
            cor += (pred[mask] == lab[mask]).sum().item()
            n_tok += mask.sum().item()
        return tot/n_tok, cor/n_tok

    val_loss_v9p, val_acc_v9p = eval_mlm(model_v9p, val_loader_p)
    print(f"V9' final val: L_mlm={val_loss_v9p:.4f}, acc={val_acc_v9p:.4f}")

    torch.save({
        'config': cfg_v9p.__dict__,
        'state_dict': model_v9p.state_dict(),
        'history': model_v9p.history,
        'val_mlm_loss': val_loss_v9p,
        'val_mlm_acc':  val_acc_v9p,
        'n_heads_final': sum(len(ls.heads) for ls in model_v9p.layer_states),
        'n_layers_final': len(model_v9p.layer_states),
        'heads_per_layer_final': [len(ls.heads) for ls in model_v9p.layer_states],
        'tokenizer_vocab': tokenizer.vocab,
        'corpus': 'ensembl_multi_vertebrate_8species',
        'ensembl_release': ENSEMBL_RELEASE,
        'species_list': [s[1] for s in SPECIES],
        'n_train_seqs': len(train_seqs),
        'n_val_seqs': len(val_seqs),
    }, os.path.join(PHASE11_DIR, 'incrt_geo_protein_encoder.pt'))

    status['phase11_done'] = True
    save_status(status)
    log_event(f"Phase 11 DONE — V9' val L_mlm={val_loss_v9p:.4f}, acc={val_acc_v9p:.4f}, "
              f"n_layers={len(model_v9p.layer_states)}, "
              f"heads_per_layer={[len(ls.heads) for ls in model_v9p.layer_states]}")


In [ ]:
# Quick diagnostic plots after Phase 2
v9_ckpt = os.path.join(PHASE5_DIR, 'incrt_geo_protein_encoder.pt')
if os.path.exists(v9_ckpt):
    ck = torch.load(v9_ckpt, map_location='cpu', weights_only=False)
    h = ck['history']
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('V9 Pretraining Diagnostics (1-mer, n=512)', fontsize=12)
    if h.get('loss_mlm'):
        w = 50
        sm = np.convolve(h['loss_mlm'], np.ones(w)/w, mode='valid')
        axes[0,0].plot(sm); axes[0,0].set_title('MLM loss (smoothed)')
        axes[0,0].axhline(np.log(8005), color='red', ls=':', alpha=0.6)
        axes[0,0].grid(alpha=0.3)
    if h.get('loss_alpha'):
        sm = np.convolve(h['loss_alpha'], np.ones(50)/50, mode='valid')
        axes[0,1].plot(sm, color='purple'); axes[0,1].set_title('L_alpha')
        axes[0,1].grid(alpha=0.3)
    if h.get('Pa_norm_total'):
        axes[0,2].plot(h['Pa_norm_total'], color='orange')
        axes[0,2].set_yscale('log'); axes[0,2].set_title('||P_a||^2')
        axes[0,2].grid(alpha=0.3, which='both')
    if h.get('n_heads'):
        axes[1,0].step(range(len(h['n_heads'])), h['n_heads'], color='green')
        axes[1,0].set_title('# heads (cumulative)'); axes[1,0].grid(alpha=0.3)
    if h.get('n_layers'):
        axes[1,1].step(range(len(h['n_layers'])), h['n_layers'], color='darkgreen')
        axes[1,1].set_title('# layers'); axes[1,1].grid(alpha=0.3)
    if h.get('alpha_top_layer'):
        axes[1,2].plot(h['alpha_top_layer'], color='darkviolet')
        axes[1,2].axhline(0.9, color='red', ls='--', alpha=0.6)
        axes[1,2].set_title('alpha_top'); axes[1,2].set_ylim(0.5, 1.05)
        axes[1,2].grid(alpha=0.3)
    plt.tight_layout()
    fig_path = os.path.join(PHASE5_DIR, 'v9_diagnostics.png')
    plt.savefig(fig_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')
    print(f'Level-3 events: {h.get("level3_events", [])}')


## PHASE 12 — Pfam-50 evaluation on v9' checkpoint

Identical Pfam-50 protocol (linear probe + light fine-tune) as Phase 6/8/10. The Pfam annotations are obtained from the UniProt REST API and the test split is deterministic with seed 42, so accuracies are directly comparable to v8/v9/v10/v11.

In [ ]:
status = load_status()
if status.get('phase12_done', False):
    print('Phase 12 already done. Loading existing summary...')
    with open(os.path.join(PHASE12_DIR, 'pfam_summary.json')) as f:
        phase12_summary = json.load(f)
    print(json.dumps(phase12_summary, indent=2))
else:
    log_event("Phase 12 START — Pfam evaluation on v9' (SwissProt-pretrained)")
    v9p_ckpt = os.path.join(PHASE11_DIR, 'incrt_geo_protein_encoder.pt')
    if not os.path.exists(v9p_ckpt):
        raise FileNotFoundError(f"V9' checkpoint not found at {v9p_ckpt}.")
    phase12_summary = evaluate_pfam(v9p_ckpt, PHASE12_DIR, top_n=50, label='v9p_swissprot')
    status = load_status()
    status['phase12_done'] = True
    save_status(status)
    log_event('Phase 12 DONE')


## Final summary — v9 vs the existing baselines

Combine the new v9 result with v6/v7/v8 results from earlier phases and ESM-2/ProtBERT baselines from Phase 4.

In [ ]:
# Build the full comparison
import pandas as pd

rows = []
rows.append({
    'Model': 'Random baseline',
    'Parameters': '—',
    'Probe test acc': f'{100/50:.2f}%', 'Probe top-5': f'{500/50:.2f}%',
})

# Load all available summaries
def safe_load(path):
    if os.path.exists(path):
        with open(path) as f: return json.load(f)
    return None

baselines = safe_load(os.path.join(PIPELINE_ROOT, 'phase4_baselines/baselines_summary.json'))
if baselines:
    rows.append({
        'Model': f'ESM-2 small ({baselines["esm2_8m"]["n_params"]/1e6:.1f}M)',
        'Parameters': f'{baselines["esm2_8m"]["n_params"]:,}',
        'Probe test acc': f'{baselines["esm2_8m"]["test_acc"]*100:.2f}%',
        'Probe top-5':    f'{baselines["esm2_8m"]["test_top5"]*100:.2f}%',
    })
    rows.append({
        'Model': f'ProtBERT ({baselines["protbert"]["n_params"]/1e6:.0f}M)',
        'Parameters': f'{baselines["protbert"]["n_params"]:,}',
        'Probe test acc': f'{baselines["protbert"]["test_acc"]*100:.2f}%',
        'Probe top-5':    f'{baselines["protbert"]["test_top5"]*100:.2f}%',
    })

# Old INCRT-geo runs
prev_v8 = safe_load(os.path.join(PIPELINE_ROOT, 'phase3_pfam_v8/pfam_summary.json'))
if prev_v8:
    rows.append({
        'Model': f'INCRT-geo v8 (3-mer, n=256)',
        'Parameters': f'{prev_v8["architecture"]["total_params"]:,}',
        'Probe test acc': f'{prev_v8["linear_probe"]["test_acc"]*100:.2f}%',
        'Probe top-5':    f'{prev_v8["linear_probe"]["test_top5"]*100:.2f}%',
    })

# v9 (this run)
v9_summary = safe_load(os.path.join(PHASE6_DIR, 'pfam_summary.json'))
if v9_summary:
    rows.append({
        'Model': f'INCRT-geo v9 (1-mer, n=512) — this work',
        'Parameters': f'{v9_summary["architecture"]["total_params"]:,}',
        'Probe test acc': f'{v9_summary["linear_probe"]["test_acc"]*100:.2f}%',
        'Probe top-5':    f'{v9_summary["linear_probe"]["test_top5"]*100:.2f}%',
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(os.path.join(PIPELINE_ROOT, 'final_comparison_v9.csv'), index=False)
print(f'\nSaved: {os.path.join(PIPELINE_ROOT, "final_comparison_v9.csv")}')

print('\n=== V9 PIPELINE COMPLETE ===')
log_event('V9 PIPELINE COMPLETE')
